# Proof of Concept : Génération de liste de courses à partir d'une recette

Ce POC utilise un LLM via **LangChain v2** avec **LCEL**, connecté à un serveur **LMStudio**, pour générer une liste structurée de courses à partir d'une série de recettes.

## Objectif
- Identifier les ingrédients de chaque recette
- Regrouper les ingrédients identiques et cumuler leurs quantités
- Classer les ingrédients selon une hiérarchie prédéfinie
- Produire un JSON structuré en sortie contenant la liste de courses

## 1. Préambule : Installation & Imports

Installation et imports nécessaires pour exécuter le POC.

In [ ]:
# Installation des dépendances (à exécuter une seule fois)
!pip install langchain
!pip install langchain-community
!pip install langchain-core
!pip install pydantic
!pip install python-dotenv

In [67]:
# Imports nécessaires
from langchain_core.runnables.passthrough import RunnablePassthrough
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional, Any
from datetime import datetime
import json

## 2. Configuration : Modèle & Prompt

Initialisation du modèle LLM via LMStudio et définition du prompt.

In [ ]:
# Configuration du modèle 
model_name = "ministral-3-14b-instruct-2512"
api_base = "http://100.121.195.119:1234/v1"
temperature = 0.0

model = ChatOpenAI(
    model=model_name,
    openai_api_base=api_base,
    openai_api_key="not-needed",
    temperature=temperature
)

# Configuration du parser
# parser = StrOutputParser

# Prompt principal
prompt_template = '''
Tu es un assistant qui doit réaliser une liste de courses de tous les ingrédients présents dans les recettes.  Pour cela, tu devras 
    • Identifier les différentes recettes
    • considérer que des récettes qui apparaissent plusieurs fois sont des recettes diffférentes (ne pas mutualiser les ingrédients)
    • Pour chaque recette identifiée, identifier les ingrédients de chaque recette
    • Identifier chaque ingrédient identique et les regrouper. 

Ensuite tu devras présenter les ingrédients en respectant les catégories suivantes et l'ordre :
{categorie}
puis les sous catégories suivantes

Pour ta réponse présente uniquement (et rien d'autre): 
    • Les recettes listées
    • Les catégories et dans chaque catégorie les différents ingrédients en respectant l'ordre présenté précédemment. Pour chaque ingrédient, commencer par un point d'énumération, le nom de l'ingrédient, une tabulation, la parenthèse avec la quantité requise pour toutes les recettes en mentionnant le nom des recettes.

Tu devras challenger ton résultat en relisant chaque recette, chaque ingrédient et vérifier que la quantité est bein présente entre paranthèse, qu'il n'y a pas de mutalsisation entre recettes. Tu devras également relire ta réponse et vérifier que les ingrédients identiques ont bien été regroupés, qu'ils n'apparaissent pas plusieurs fois.

Recettes : {recettes}
'''


## 3. Définition des constantes

Définition des catégories et ordre pour la classification.

In [ ]:
# Constantes
CATEGORIES = [
    "viandes",
    "saucisserie",
    "poissons_crustaces",
    "legumes",
    "fruits",
    "laitages",
    "herbes_condiments",
    "farines_pains_cereales",
    "sucres_levures",
    "bouillons_fonds",
    "boissons",
    "autres"
]

# # Ordre des sous-catégories pour chaque catégorie
# ORDRE_SAUCISSERIE = ["jambon", "lardons"]
# ORDRE_POISSONS = ["poissons", "crevettes"]
# ORDRE_LEGUMES = ["ail", "oignons", "pommes_de_terre"]
# ORDRE_LAITAGES = ["lait", "beurre", "creme", "fromage", "oeufs"]


## 4. Définition des classes utilitaires

Classes Pydantic pour la structure de sortie.

In [71]:
# Structure entrée

class Ingredient(BaseModel):
    quantite: Optional[str] = None
    unite: Optional[str] = None
    ingredient: str
    
class RecetteSelection(BaseModel):
    id_recette: str
    nb_recette: int
    nom_livre: str
    numero_livre: Optional[str]
    nom_recette: str
    type_recette: str
    liste_ingredients: List[Ingredient]

# Structure de la sortie
class ListeCourse(BaseModel):
    date_liste_course : datetime = Field(description="Date de la liste des courses")
    liste_recette: List[RecetteSelection] = Field(description="Liste des recettes associées à la liste de course")
    liste_course: str = Field(description="Liste des courses")
                                                  


## 5. Fonctions utilitaires

Fonctions d'analyse et de traitement des recettes.

In [ ]:
from typing import List
from flask import session


# Fonction de prétraitement des recettes pour le LLM 
def preparer_recettes(recettes: List[RecetteSelection]) -> str:
    lignes: List[str] = []
    for r in recettes:
        for _ in range(r.nb_recette):
            ingredients = ", ".join(f"{ing.quantite} {ing.unite} {ing.ingredient}" for ing in r.liste_ingredients)
            lignes.append(f"{r.nom_recette} : {ingredients}")
    return "\n".join(lignes)
    

# Fonction pour construire la liste des courses par le LLM
def construire_liste_course(recettes: List[RecetteSelection]) -> str:

    prompt = PromptTemplate(
        template=prompt_template,
        input_variables=["recettes"],
        partial_variables={"categorie":CATEGORIES}
    )

    # # Debug propre
    # print(preparer_recettes(recettes))
    # print(type(preparer_recettes(recettes)))

    chain = (
        {
            "recettes": RunnableLambda(preparer_recettes),
            "categorie":lambda _: CATEGORIES,
        }
        | prompt
        | model
        | StrOutputParser()
    )

    # Appel correct : un dict avec la clé "recettes"
    response: str = chain.invoke(recettes)
    return response

# Fonction pour construire l'objet pydantic de sortie
def construire_output_pydantic(
    recettes: List[RecetteSelection],
    liste_courses: str
    ) -> ListeCourse:
    
    return ListeCourse(
        date_liste_course=datetime.now(),
        liste_recette=recettes,
        liste_course=liste_courses
    )


# Fonction pour construire l'objet session de sortie (Flask)
def construire_output_session(
    recettes: List[RecetteSelection],
    liste_courses: str
) -> None:

    # 1) Convertir en dict pour la session
    data_dict = construire_output_pydantic(recettes,liste_courses).model_dump()

    # # 2) Debug (optionnel)
    # print("DEBUG - liste_course stockée dans session :", data_dict)

    # 3) Stockage dans la session
    session["liste_course"] = data_dict


    # Comment récupérer l’objet dans un autre module ?
    # data = session.get("liste_course")
    # if data:
    #     liste_course = ListeCourse(**data)



## 6. Logique principale

Exécution du traitement avec LLM.

In [ ]:
# Exemple de données d'entrée
recettes = [
    RecetteSelection(
        id_recette="1",
        nb_recette=1,
        nom_livre="Cuisine Traditionnelle",
        numero_livre="2",
        nom_recette="Poulet au vin rouge",
        type_recette="plat principal",
        liste_ingredients=[
            {"ingredient": "poulet", "quantite": "1"},
            {"ingredient": "vin rouge", "quantite": "1 bouteille"},
            {"ingredient": "oignons", "quantite": "3"},
            {"ingredient": "ail", "quantite": "4 gousses"},
        ]
    ),
    RecetteSelection(
        id_recette="2",
        nb_recette=2,
        nom_livre="Cuisine Traditionnelle",
        numero_livre="2",
        nom_recette="Salade César",
        type_recette="entrée",
        liste_ingredients=[
            {"ingredient": "laitue", "quantite": "1"},
            {"ingredient": "tomate", "quantite": "2"},
            {"ingredient": "oignons", "quantite": "200 grammes"},
            {"ingredient": "ail", "quantite": "2 gousses"},
        ]
    )
]


# Traitement principal
resultat = construire_liste_course(recettes)
Liste_course_pydantic = construire_output_pydantic(recettes,resultat)

## 7. Test du POC

Affichage du résultat de la liste de courses.

In [ ]:
# Test et affichage
print("=== Résultat du traitement ===")
print(resultat)

# Création d'une structure JSON de sortie
json_output = {
    "id_recette": "1",
    "nb_recette": 1,
    "nom_livre": "Cuisine Traditionnelle",
    "numero_livre": "2",
    "nom_recette": "Poulet au vin rouge",
    "type_recette": "plat principal",
    "liste_course": resultat.split("\n")
}

print("\n=== Structure JSON ===")
print(json.dumps(json_output, indent=2))

## 8. Interface utilisateur (Terminal)

Interface de test en ligne de commande.

In [ ]:
# Interface utilisateur
print("=== Test POC : Liste de Courses ===")
print("Recettes fournies :")
for r in recettes:
    print(
        f"- {r.nom_recette} ({r.type_recette}) : "
        f"{', '.join(f'{ing.quantite} {ing.unite} {ing.ingredient}' for ing in r.liste_ingredients)}"
    )


print("\nRésultat de la liste de courses générée par le LLM :")
print(resultat)

## 9. Résumé des capacités du POC

- **Analyse de recettes** via LLM
- **Regroupement d'ingrédients identiques** et **cumul des quantités**
- **Classement hiérarchique** selon une logique prédéfinie
- **Génération de JSON structuré** avec sortie standardisée
- **Validation de la réponse par le LLM lui-même** (challenging)

### Limites actuelles :
- Dépendance à un serveur LMStudio en ligne
- Nécessité d’un prompt très détaillé pour structuration correcte
- Risque de mauvaise interprétation si les unités ne sont pas cohérentes

Ce POC peut être étendu avec une base de données et une interface graphique.